In [ ]:
import pandas as pd
import numpy as np

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval
z = 1.96
lower_ci = mean_demand - z * std_error
upper_ci = mean_demand + z * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval using t-score
alpha = 0.05
# t critical value depends on df = count - 1
t_values = stats.t.ppf(1 - alpha/2, df=count - 1)

lower_ci = mean_demand - t_values * std_error
upper_ci = mean_demand + t_values * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# =========================
# CONFIGURATION
# =========================

ROLLING_DAYS = 7
SERVICE_LEVEL = 0.95   # Change to 0.90, 0.95 etc.
MACHINE_CAPACITY = 5000

# =========================
# LOAD FILE
# =========================

df = pd.read_excel("your_file.xlsx")

# =========================
# WIDE → LONG
# =========================

df_long = df.melt(
    id_vars=["Material"],
    var_name="Date",
    value_name="Demand"
)

df_long["Date"] = (
    df_long["Date"]
    .str.replace(" Total Production Plan", "", regex=False)
)

df_long["Date"] = pd.to_datetime(df_long["Date"])

df_long = df_long.sort_values(["Material", "Date"])

# =========================
# ROLLING MEAN & STD
# =========================

df_long["Mean_Demand"] = (
    df_long.groupby("Material")["Demand"]
    .rolling(ROLLING_DAYS)
    .mean()
    .reset_index(level=0, drop=True)
)

df_long["Std_Demand"] = (
    df_long.groupby("Material")["Demand"]
    .rolling(ROLLING_DAYS)
    .std()
    .reset_index(level=0, drop=True)
)

df_long["Std_Demand"] = df_long["Std_Demand"].fillna(0)

# =========================
# SAFE PRODUCTION
# =========================

Z = norm.ppf(SERVICE_LEVEL)

df_long["Safe_Production_Qty"] = (
    df_long["Mean_Demand"]
    + Z * df_long["Std_Demand"]
)

# Avoid negative or NaN
df_long["Safe_Production_Qty"] = df_long["Safe_Production_Qty"].fillna(0)
df_long["Safe_Production_Qty"] = df_long["Safe_Production_Qty"].apply(lambda x: max(0, x))

# Apply capacity limit
df_long["Final_Production"] = df_long["Safe_Production_Qty"].apply(
    lambda x: min(x, MACHINE_CAPACITY)
)

# =========================
# SAVE OUTPUT
# =========================

df_long.to_excel("safe_production_output.xlsx", index=False)

print("Safe production plan generated.")
